In [1]:
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import csr_matrix

In [12]:
ratings_df=pd.read_csv('Ratings.csv')



user_rating_counts = ratings_df['User-ID'].value_counts()
active_users = user_rating_counts[user_rating_counts >= 200].index
ratings_filtered = ratings_df[ratings_df['User-ID'].isin(active_users)]

print(f"Users before filtering: {len(user_rating_counts)}")
print(f"Users after filtering (200+ ratings): {len(active_users)}")
print(f"Ratings before filtering: {len(ratings_df)}")
print(f"Ratings after user filtering: {len(ratings_filtered)}")

# Count ratings per book
book_rating_counts = ratings_filtered['ISBN'].value_counts()

# Filter books with at least 100 ratings
popular_books = book_rating_counts[book_rating_counts >= 100].index
ratings_filtered = ratings_filtered[ratings_filtered['ISBN'].isin(popular_books)]

print(f"Books with 100+ ratings: {len(popular_books)}")
print(f"Final ratings count: {len(ratings_filtered)}")

Users before filtering: 105283
Users after filtering (200+ ratings): 905
Ratings before filtering: 1149780
Ratings after user filtering: 527556
Books with 100+ ratings: 100
Final ratings count: 13793


In [14]:
books_df=pd.read_csv('Books.csv')
# Merge ratings with book titles to get full book information
book_info = ratings_filtered.merge(books_df[['ISBN', 'Book-Title']], on='ISBN')

# Create pivot table: rows=books, columns=users, values=ratings
book_pivot_table = book_info.pivot_table(
    index='Book-Title',
    columns='User-ID',
    values='Book-Rating'
)

# Fill NaN values with 0 (no rating = 0)
book_pivot_table.fillna(0, inplace=True)

print(f"\nUser-Item Matrix shape: {book_pivot_table.shape}")
print(f"Number of books: {book_pivot_table.shape[0]}")
print(f"Number of users: {book_pivot_table.shape[1]}")

C:\Users\adars\AppData\Local\Temp\ipykernel_24060\830031669.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books_df=pd.read_csv('Books.csv')



User-Item Matrix shape: (99, 857)
Number of books: 99
Number of users: 857


In [15]:
# Convert the pivot table to a sparse matrix for efficiency
book_pivot_sparse = csr_matrix(book_pivot_table.values)

# Calculate and display sparsity
sparsity = 1 - (book_pivot_sparse.nnz / (book_pivot_sparse.shape[0] * book_pivot_sparse.shape[1]))
print(f"\nMatrix sparsity: {sparsity:.2%}")
print(f"Non-zero elements: {book_pivot_sparse.nnz:,}")

# Train the KNN model
# metric='cosine' - measures angle between rating vectors
# algorithm='brute' - exact computation (slower but guaranteed accuracy)
# n_neighbors=6 - get 6 neighbors (book itself + 5 recommendations)
model = NearestNeighbors(metric='cosine', algorithm='brute', n_neighbors=6)
model.fit(book_pivot_sparse)

print("\n✓ KNN model trained successfully!")
print(f"Model ready to find recommendations for {len(book_pivot_table)} books")


Matrix sparsity: 95.79%
Non-zero elements: 3,568

✓ KNN model trained successfully!
Model ready to find recommendations for 99 books


In [16]:
def get_recommends(book_title):
   
    
    # Check if the book exists in the dataset
    if book_title not in book_pivot_table.index:
        raise ValueError(f"'{book_title}' not found in dataset")
    
    # Find the row index of the book
    book_index = book_pivot_table.index.get_loc(book_title)
    
    # Get the book's rating vector (as a sparse matrix)
    book_vector = csr_matrix(book_pivot_table.iloc[book_index].values)
    
    # Find the 6 nearest neighbors (including the book itself)
    distances, indices = model.kneighbors(book_vector, n_neighbors=6)
    
    # Convert arrays to 1D (flatten) and skip the first result (the book itself)
    distances = distances.flatten()[1:]  # Skip the book itself
    indices = indices.flatten()[1:]      # Skip the book itself
    
    # Build the recommendations list
    recommendations = []
    for i, distance in zip(indices, distances):
        recommendations.append([book_pivot_table.index[i], distance])
    
    # Return in the required format
    return [book_title, recommendations]



In [24]:
test_title = book_pivot_table.index[0]
result = get_recommends(test_title)

print(f"\nRecommendations for: {result[0]}\n")
print("Top 5 Similar Books:")
print("-" * 70)
for i, (book, distance) in enumerate(result[1], 1):
    print(f"{i}. {book}")
    print(f"   Distance: {distance}")
    print()

print("\nFull result object:")
print(result)

# Additional test with a different book
print("\n" + "=" * 70 + "\n")
print("Testing with another book from the dataset:")
test_book_2 = book_pivot_table.index[10]  # Get a different book
result_2 = get_recommends(test_book_2)

print(f"\nRecommendations for: {result_2[0]}\n")
for i, (book, distance) in enumerate(result_2[1], 1):
    print(f"{i}. {book} (distance: {distance:.4f})")


Using closest match: 1st to Die: A Novel

Recommendations for: 1st to Die: A Novel

Top 5 Similar Books:
----------------------------------------------------------------------
1. Along Came a Spider (Alex Cross Novels)
   Distance: 0.6923980317962177

2. Two for the Dough
   Distance: 0.758241671664506

3. One for the Money (Stephanie Plum Novels (Paperback))
   Distance: 0.7706038246057285

4. Three To Get Deadly : A Stephanie Plum Novel (A Stephanie Plum Novel)
   Distance: 0.7839009208923955

5. Kiss the Girls
   Distance: 0.7923405078589497


Full result object:
('1st to Die: A Novel', [('Along Came a Spider (Alex Cross Novels)', np.float64(0.6923980317962177)), ('Two for the Dough', np.float64(0.758241671664506)), ('One for the Money (Stephanie Plum Novels (Paperback))', np.float64(0.7706038246057285)), ('Three To Get Deadly : A Stephanie Plum Novel (A Stephanie Plum Novel)', np.float64(0.7839009208923955)), ('Kiss the Girls', np.float64(0.7923405078589497))])


Testing with anoth